In [2]:
! pip install qdrant-client sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.9/258.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.64.1
    Uninstalling grpcio-1.64.1:
      Successfully

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
client = QdrantClient(":memory:")

items = [
    "London", "Paris", "New York", "Tokyo", "Berlin", "Moscow", "Beijing", "Madrid", "Rome", "Vienna",
    "Los Angeles", "Sydney", "Chicago", "Hong Kong", "Toronto", "Istanbul", "Bangkok", "Dubai", "San Francisco",
    "Munich", "Miami", "Amsterdam", "Zurich", "Melbourne", "Sao Paulo", "Singapore", "Brussels", "Milan",
    "Buenos Aires", "Mexico City", "Seoul", "Stockholm", "Jakarta", "Kuala Lumpur", "Lisbon", "Copenhagen",
    "Athens", "Dublin", "Budapest", "Warsaw", "Oslo", "Helsinki", "Prague", "Cape Town", "Brisbane", "Rio de Janeiro",
    "Tel Aviv", "Vancouver", "Auckland", "Hamburg", "Johannesburg", "Lima", "Santiago", "Manila", "Bogota",
    "Warsaw", "Montreal", "Kyoto", "Shanghai", "Perth", "Kolkata", "Hanoi", "Dubai", "Tunis", "Istanbul",
    "Belgrade", "Minsk", "Tbilisi", "Bucharest", "Casablanca", "Doha", "Caracas", "Riyadh", "Sofia",
    "Bangalore", "Chennai", "Guangzhou", "Lagos", "Tehran", "Cairo", "Havana", "Baghdad", "Algiers",
    "Damascus", "Quito", "Caracas", "Tashkent", "Islamabad", "Kathmandu", "Tripoli", "Kinshasa", "Amman",
    "Tirana", "Skopje", "Ljubljana", "Sarajevo", "Podgorica", "Vilnius", "Tallinn", "Riga", "Chisinau",
    "Tegucigalpa", "Panama City", "San Salvador", "San Jose", "Managua", "Guatemala City", "Port-au-Prince",
    "La Paz", "Sucre", "Baku", "Yerevan", "Ashgabat", "Nassau", "Georgetown", "Paramaribo", "Castries",
    "Bridgetown", "Kingstown", "Basseterre", "Roseau", "Saint John's", "Road Town", "Charlotte Amalie",
    "Philipsburg", "Gustavia", "Oranjestad", "Fort-de-France", "St. George's", "Kingstown", "Castries",
    "Port of Spain", "Saint-Pierre", "Saint Denis","Apple", "Banana", "Orange", "Mango", "Grapes",
    "Pineapple", "Strawberry", "Blueberry", "Papaya", "Watermelon"
]

item_vectors = model.encode(items).tolist()

collection_name = "anomalous_city_collection2"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=len(item_vectors[0]), distance=models.Distance.COSINE)
)

points = [
    models.PointStruct(id=i, vector=vector, payload={"name": item})
    for i, (vector, item) in enumerate(zip(item_vectors, items))
]
client.upsert(collection_name=collection_name, points=points)



UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

Assume we have a dataset which is large, and we cannot manually go through. How will we detect anomalies?

After creating vector points like above, we will take 2-3 good examples that we *know* are good examples.

We are going to then use those good examples to find points that are most 'dissimilar' (or far apart) from those points. So, we need to treat these examples as 'negative' points -- so that the recommendation engine finds points which are opposite to this.

Using this tactic, we will first find 2-3 anomalous points.

In [ ]:
positive_points = ['Milan', 'London', 'Berlin', 'Paris',]
positive_point_vectors = model.encode(positive_points).tolist()

response = client.recommend(
    collection_name=collection_name,
    negative=positive_point_vectors,  # Treat cities as negative examples, which means it would find objects farthest from city points
    limit=3,  # Limit the number of results
    with_payload=True,
    strategy=models.RecommendStrategy.BEST_SCORE  # Use best score vector strategy
)

for point in response:
    print(f"Anomaly: {point.payload['name']}, Score: {point.score}")


Anomaly: Watermelon, Score: -0.5674867033958435
Anomaly: Papaya, Score: -0.5698751211166382
Anomaly: Blueberry, Score: -0.5727079510688782


Now that we have found 3 anomalous points, we will use them as positive examples to discover other anomalous points.

In [ ]:
negative_points = ['Watermelon', 'Papaya', 'Blueberry',]
negative_points_vectors = model.encode(negative_points).tolist()

response = client.recommend(
    collection_name=collection_name,
    positive=negative_points_vectors,  # Treat anomalies as positive example, to discover other anomalies
    limit=10,  # Limit the number of results
    with_payload=True,
    strategy=models.RecommendStrategy.BEST_SCORE  # Use best score vector strategy
)

for point in response:
    print(f"Anomaly: {point.payload['name']}, Score: {point.score}")

Anomaly: Watermelon, Score: 0.75
Anomaly: Blueberry, Score: 0.75
Anomaly: Papaya, Score: 0.75
Anomaly: Strawberry, Score: 0.6920410990715027
Anomaly: Mango, Score: 0.6880236864089966
Anomaly: Grapes, Score: 0.6864984631538391
Anomaly: Pineapple, Score: 0.6846863627433777
Anomaly: Banana, Score: 0.6806163191795349
Anomaly: Orange, Score: 0.6702149510383606
Anomaly: Apple, Score: 0.6502500176429749


In [ ]:
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

df = pd.read_csv('updated_file.csv')

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
client = QdrantClient(":memory:")

collection_name = "customer_collection"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE)
)

df['combined_text'] = df[['Customer ID', 'Name', 'Surname', 'Gender',
                           'Birthdate', 'Transaction Amount',
                           'Date', 'Merchant Name', 'Category']].astype(str).agg(' '.join, axis=1)

item_vectors = model.encode(df['combined_text'].tolist()).tolist()

points = [
    models.PointStruct(
        id=i,
        vector=vector,
        payload={
            "Customer ID": df.at[i, 'Customer ID'],
            "Name": df.at[i, 'Name'],
            "Surname": df.at[i, 'Surname'],
            "Gender": df.at[i, 'Gender'],
            "Birthdate": df.at[i, 'Birthdate'],
            "Transaction Amount": df.at[i, 'Transaction Amount'],
            "Date": df.at[i, 'Date'],
            "Merchant Name": df.at[i, 'Merchant Name'],
            "Category": df.at[i, 'Category'],
        }
    )
    for i, vector in enumerate(item_vectors)
]

client.upsert(collection_name=collection_name, points=points)

print("All fields have been encoded and stored in Qdrant successfully!")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All fields have been encoded and stored in Qdrant successfully!


In [7]:
positive_points = ['764762', '521807', '504238', '793150']  # Your provided positive points
positive_point_vectors = model.encode(positive_points).tolist()

# Load customer data from Qdrant
scroll_response, next_page = client.scroll(collection_name=collection_name, limit=20000)  # Unpack scroll response

# Extract customer IDs and names from the scroll response
customer_data = {}
for point in scroll_response:  # Iterate through the list of points
    customer_id = point.payload['Customer ID']
    name = point.payload['Name']
    customer_data[customer_id] = name

# Implement recommendation using the provided positive points
response = client.recommend(
    collection_name=collection_name,
    negative=positive_point_vectors,
    limit=10,
    with_payload=True,
    strategy=models.RecommendStrategy.BEST_SCORE
)

# Print results with customer IDs
for point in response:
    customer_id = point.payload['Customer ID']  # Fetch the customer ID from the payload
    name = customer_data.get(customer_id, "Unknown")  # Get name or use "Unknown"
    print(f"Anomaly: {name} (Customer ID: {customer_id}), Score: {point.score}")

Anomaly: William (Customer ID: 183690), Score: -0.5663217306137085
Anomaly: Sydney (Customer ID: 978688), Score: -0.576812744140625
Anomaly: Sierra (Customer ID: 186790), Score: -0.5769281387329102
Anomaly: Brendan (Customer ID: 978824), Score: -0.5779920816421509
Anomaly: Deborah (Customer ID: 978221), Score: -0.5782595872879028
Anomaly: Benjamin (Customer ID: 978197), Score: -0.5802853107452393
Anomaly: Eric (Customer ID: 190533), Score: -0.5809781551361084
Anomaly: Martin (Customer ID: 382033), Score: -0.5811682939529419
Anomaly: Loretta (Customer ID: 1995), Score: -0.581832230091095
Anomaly: Travis (Customer ID: 51), Score: -0.5828118324279785


In [8]:
positive_points = ['ndi6692', 'ndi9325', 'ndi6493', 'ndi2469']  # Your provided positive points
positive_point_vectors = model.encode(positive_points).tolist()

# Load customer data from Qdrant
scroll_response, next_page = client.scroll(collection_name=collection_name, limit=20000)  # Unpack scroll response

# Extract customer IDs and names from the scroll response
customer_data = {}
for point in scroll_response:  # Iterate through the list of points
    customer_id = point.payload['Customer ID']
    name = point.payload['Name']
    customer_data[customer_id] = name

# Implement recommendation using the provided positive points
response = client.recommend(
    collection_name=collection_name,
    positive=positive_point_vectors,
    limit=10,
    with_payload=True,
    strategy=models.RecommendStrategy.BEST_SCORE
)

# Print results with customer IDs
for point in response:
    customer_id = point.payload['Customer ID']  # Fetch the customer ID from the payload
    name = customer_data.get(customer_id, "Unknown")  # Get name or use "Unknown"
    print(f"Anomaly: {name} (Customer ID: {customer_id}), Score: {point.score}")

Anomaly: Brooke (Customer ID: 645298), Score: 0.680458664894104
Anomaly: Nicholas (Customer ID: ndi7876), Score: 0.680246889591217
Anomaly: Daniel (Customer ID: 120216), Score: 0.6795614361763
Anomaly: Unknown (Customer ID: 880423), Score: 0.6773449182510376
Anomaly: Lisa (Customer ID: 280701), Score: 0.6772925853729248
Anomaly: Unknown (Customer ID: 283649), Score: 0.6758548021316528
Anomaly: Christian (Customer ID: 667841), Score: 0.6750497221946716
Anomaly: Unknown (Customer ID: 447790), Score: 0.6747046709060669
Anomaly: Daniel (Customer ID: 223392), Score: 0.6746610403060913
Anomaly: Jason (Customer ID: 680473), Score: 0.6746330261230469
